# Tugas Week 3 – Simulasi Fuzzy Sugeno
**Kasus:** Sistem peringatan jarak aman layar monitor  
**Metode:** Fuzzy Sugeno orde-0  
**Input:** Jarak wajah ke layar (cm) dan intensitas cahaya ruangan (lux)  
**Output:** Tingkat risiko mata (0–100)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Fungsi keanggotaan (membership function)

In [2]:
def turun(x, a, b):
    return np.clip((b - x) / (b - a), 0, 1)

def naik(x, a, b):
    return np.clip((x - a) / (b - a), 0, 1)

def segitiga(x, a, b, c):
    return np.maximum(np.minimum((x - a) / (b - a), (c - x) / (c - b)), 0)

# Input 1: jarak (cm)
def jarak_mf(x):
    return {"Dekat": turun(x, 25, 45), "Sedang": segitiga(x, 30, 50, 70), "Jauh": naik(x, 55, 75)}

# Input 2: cahaya (lux)
def cahaya_mf(x):
    return {"Gelap": turun(x, 50, 150), "Normal": segitiga(x, 100, 250, 400), "Terang": naik(x, 350, 500)}

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
xj = np.linspace(0, 100, 500)
for nama, y in jarak_mf(xj).items(): ax[0].plot(xj, y, label=nama, lw=2)
ax[0].set(title="Input 1: Jarak", xlabel="cm", ylabel="Derajat keanggotaan"); ax[0].legend(); ax[0].grid(alpha=.3)

xc = np.linspace(0, 600, 600)
for nama, y in cahaya_mf(xc).items(): ax[1].plot(xc, y, label=nama, lw=2)
ax[1].set(title="Input 2: Cahaya", xlabel="lux"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 2. Aturan fuzzy (9 rule) dan output Sugeno
Output tiap aturan berupa konstanta (singleton), lalu dihitung dengan rata-rata terbobot:  
$$z = \frac{\sum w_i z_i}{\sum w_i}, \quad w_i = \min(\mu_{jarak}, \mu_{cahaya})$$

In [4]:
aturan = pd.DataFrame(
    [[100, 80, 60], [60, 30, 20], [30, 5, 0]],
    index=["Dekat", "Sedang", "Jauh"], columns=["Gelap", "Normal", "Terang"])
aturan.index.name = "Jarak \\ Cahaya"
aturan

,Gelap,Normal,Terang
Jarak \ Cahaya,,,
Dekat,100,80,60
Sedang,60,30,20
Jauh,30,5,0


In [5]:
def fuzzy_sugeno(jarak, cahaya):
    mj, mc = jarak_mf(jarak), cahaya_mf(cahaya)
    num = den = 0.0
    for j in aturan.index:
        for c in aturan.columns:
            w = min(mj[j], mc[c])          # operator AND = min
            num += w * aturan.loc[j, c]
            den += w
    return num / den if den > 0 else 0.0

def status(r):
    return "Aman" if r < 30 else ("Waspada" if r < 65 else "Bahaya")

## 3. Simulasi skenario

In [6]:
skenario = [(20, 30), (20, 300), (40, 80), (50, 250), (60, 450), (80, 250), (90, 500)]
hasil = pd.DataFrame(
    [(j, c, round(fuzzy_sugeno(j, c), 1), status(fuzzy_sugeno(j, c))) for j, c in skenario],
    columns=["Jarak (cm)", "Cahaya (lux)", "Risiko", "Status"])
hasil

,Jarak (cm),Cahaya (lux),Risiko,Status
0,20,30,100.0,Bahaya
1,20,300,80.0,Bahaya
2,40,80,73.3,Bahaya
3,50,250,30.0,Waspada
4,60,450,13.3,Aman
5,80,250,5.0,Aman
6,90,500,0.0,Aman


## 4. Permukaan output (hubungan 2 input terhadap risiko)

In [7]:
J, C = np.meshgrid(np.linspace(10, 100, 60), np.linspace(0, 600, 60))
Z = np.vectorize(fuzzy_sugeno)(J, C)

fig = plt.figure(figsize=(12, 4.8))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(J, C, Z, cmap="RdYlGn_r")
ax1.set(xlabel="Jarak (cm)", ylabel="Cahaya (lux)", zlabel="Risiko", title="Surface")
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(J, C, Z, levels=20, cmap="RdYlGn_r"); plt.colorbar(cs, ax=ax2, label="Risiko")
ax2.set(xlabel="Jarak (cm)", ylabel="Cahaya (lux)", title="Contour")
plt.tight_layout(); plt.show()

## 5. Kesimpulan
- Semakin dekat jarak dan semakin gelap ruangan, nilai risiko semakin tinggi.
- Sistem tidak memakai batas tegas (statis), sehingga perubahan status berlangsung halus.
- Logika yang sama dipakai pada firmware ESP32 untuk proyek PiL.